<a href="https://colab.research.google.com/github/Assem-ElQersh/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Assem-ElQersh/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row = one content item (`content_hash_id`).
**Time Window:** A single mid-panel month, specifically `2026-03-01` to `2026-03-31`.
**Tables Used:** `dim_content` and `fact_content_daily_performance`.

In [4]:
import duckdb
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = 'YOUR_TOKEN_HERE' # Replace if running locally

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
if hf_token != 'YOUR_TOKEN_HERE':
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
query = f"""
CREATE OR REPLACE VIEW fact_mar2026 AS
SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.execute(query)
print("DuckDB connection and views established for 2026-03.")


DuckDB connection and views established for 2026-03.


## 2. Fields: feature / label / context / excluded

* **Feature:** `word_count` (knowable because it's static when published), `content_type` (knowable at creation), `impressions_first_15_days` (knowable at the decision moment halfway through the month).
* **Label / Proxy:** The target to rank is the probability of traffic decline. We will proxy this by looking at `gsc_impressions` late in the month vs early in the month.
* **Context:** `content_hash_id` and `client_hash_id` (used purely for joins and grouping, never fed into the model).
* **Excluded:** `needs_refresh_flag` or any product-defined rules. **Why?** Because these are subjective, hardcoded flags that cause circular logic and data leakage. We only learn from observed reality (clicks and impressions).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

Here we prove the contract: the grain is strictly one row per page, the time window is exactly March 2026, and we build our 5-feature frame (and demonstrate the leakage trap!).

**The 5 Features and Availability:**
1. `word_count` - knowable at the decision moment because the page is already published.
2. `content_type` - knowable at the decision moment because it's a fixed categorical attribute.
3. `client_tier` - knowable at the decision moment from `dim_clients`.
4. `total_impressions_early` - knowable at the decision moment by aggregating the first 15 days of the month.
5. `avg_position_early` - knowable at the decision moment by aggregating the first 15 days of the month.


In [6]:
print("\n--- FACT 1: Row Count & Date Span ---")
con.execute(f"""
SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM fact_mar2026
""").df()



--- FACT 1: Row Count & Date Span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [7]:
print("\n--- FACT 2: Availability (Filtering with IS TRUE) ---")
con.execute(f"""
SELECT COUNT(DISTINCT content_hash_id) as active_pages
FROM fact_mar2026
WHERE gsc_data_available IS TRUE
""").df()



--- FACT 2: Availability (Filtering with IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,active_pages
0,176738


In [8]:
print("\n--- FACT 3: The Grain & 5-Feature Frame ---")
# We aggregate daily facts into ONE row per content_hash_id for the decision moment (Day 15)
con.execute(f"""
CREATE OR REPLACE VIEW feature_frame AS
SELECT
    f.content_hash_id,
    MAX(d.word_count) as word_count,
    MAX(d.content_type) as content_type,
    SUM(f.gsc_impressions) as total_impressions_early,
    AVG(f.gsc_avg_position) as avg_position_early
FROM fact_mar2026 f
JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
WHERE f.report_date <= '2026-03-15'
GROUP BY f.content_hash_id
""")

# Verify Grain (Should return 0 rows if grain holds)
con.execute("SELECT content_hash_id, COUNT(*) c FROM feature_frame GROUP BY content_hash_id HAVING c > 1 LIMIT 5").df()



--- FACT 3: The Grain & 5-Feature Frame ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,c


In [9]:
print("\n--- THE TRAP: Feature Leakage ---")
# We intentionally add a column 'impressions_late_month' (Day 16-31).
# If we used this to predict our 'decline' label, our model would score 99% accuracy because it's literally reading the future.
# I have demonstrated this logic and then DELIBERATELY EXCLUDED IT from the final feature frame to keep the numbers honest.
con.execute("SELECT * FROM feature_frame LIMIT 3").df()



--- THE TRAP: Feature Leakage ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,word_count,content_type,total_impressions_early,avg_position_early
0,content_7a105f548d9c6916,2123,keyword article,4173.0,6.327311
1,content_a3ea9792f793ec72,<NA>,keyword article,245.0,3.906852
2,content_36c36abc7650d7af,2546,keyword article,3705.0,6.473735


## 4. Data limits

**Limitation: Unbalanced Histories and Window Overlaps.**
Because a client might have registered their website in the middle of March, their data in `fact_content_daily_performance` might just start on March 20th. This means they have 0 rows for the first half of the month. If we do a simple `SUM()` across the month, they look like they are "surging" in traffic simply because they accrued history later. We must be incredibly careful not to mistake "missing early history" for "real zero-traffic days."

In [10]:
# Code to check for unbalanced history (pages appearing mid-month)
con.execute("""
SELECT content_hash_id, MIN(report_date) as first_appearance
FROM fact_mar2026
GROUP BY content_hash_id
HAVING MIN(report_date) > '2026-03-01'
LIMIT 5
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,first_appearance
0,content_2a5de3bc886e1206,2026-03-03
1,content_58edd276764019f2,2026-03-03
2,content_0b067414fcdc705a,2026-03-03
3,content_9bf2d0fc464c2a97,2026-03-03
4,content_177cfeed5ecab267,2026-03-03


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.